In [31]:
from zcb_pricer_cir import*
from callable_zcb_pricer import*
#from bond_sensitivities import*
import matplotlib.pyplot as plt

## Parameters

In [32]:
# CIR parameters
x0 = 0.065
a = 0.1
b = 0.095
sigma = 0.1

# Yield curve for fitting
yield_curve = lambda T, a=0.05, b=0.09, c=0.2: a + b * (1 - pow(np.exp(2 * c * T), -1))

#yield_curve = lambda T, a=0.05, b=0.03, c=0.8: 0.05

# Bond parameters
N = 1
Kc = 0.95 * N
T = 1       # maturity in years
t_call_eur = 8/12 * T # European call date
tc = t_call_eur
t_call_ber = np.arange(1/12, T, 1/12)  # Bermudan monthly call dates

# Simulation parameters
n_time, n_sim = 500, 50_000

# Finite Difference parameters
Nt, Nr = 150, 200

In [ ]:
# ====================================================
# MARKET FORWARD RATE (identity-based, local bond)
# ====================================================
def f_market(t, yield_curve, eps=1e-4):
    def P_mkt(s):
        return np.exp(-s * yield_curve(s))
    Pp = P_mkt(t + eps)
    Pm = P_mkt(t - eps)
    return -(np.log(Pp) - np.log(Pm)) / (2 * eps)

# ====================================================
# CIR FORWARD RATE (state = x0 only)
# ====================================================
def f_cir(t, a, b, sigma, x0):
    h = np.sqrt(a**2 + 2 * sigma**2)
    den = 2*h + (a + h) * (np.exp(h * t) - 1)
    return (
        (2 * a * b * (np.exp(h * t) - 1)) / den
        +
        x0 * (4 * h**2 * np.exp(h * t)) / (den**2)
    )

# ====================================================
# φ(t)
# ====================================================
def phi(t, a, b, sigma, x0, yield_curve, eps=1e-5):
    return (
        f_market(t, yield_curve, eps)
        - f_cir(t, a, b, sigma, x0)
    )
    #return 0.0

# ====================================================
# shift_function = exp(∫ φ)
# ====================================================
def shift_function(t0, t1, a, b, sigma, x0, yield_curve, n=50):
    ts = np.linspace(t0, t1, n)
    vals = np.array([
        phi(t, a, b, sigma, x0, yield_curve)
        for t in ts
    ])
    return np.exp(np.trapz(vals, ts))

# ====================================================
# ANALYTICAL CIR++
# ====================================================
def zcb_cir_pp_analytical(x0, a, b, sigma, T, yield_curve, N=1.0):
    P_cir = zcb_cir_analytical(x0, a, b, sigma, T)
    sh = shift_function(0.0, T, a, b, sigma, x0, yield_curve)
    return N * sh * P_cir

# ====================================================
# MONTE CARLO CIR++
# ====================================================
def zcb_cir_pp_mc(x0, a, b, sigma, T, n_time, n_sim, yield_curve, N=1.0):
    x_paths = cir_paths(x0, a, b, sigma, T, n_time, n_sim)
    dt = T / n_time
    disc = np.exp(-np.sum(x_paths[:, :-1] * dt, axis=1))
    shift = shift_function(0.0, T, a, b, sigma, x0, yield_curve)
    return np.mean(N * shift * disc)

# ====================================================
# CRANK-NICOLSON CIR++
# ====================================================
def zcb_cir_pp_cn(x0, a, b, sigma, T, yield_curve, Nt=150, Nr=200, N=1.0):
    x_grid = np.linspace(0, b + 5 * sigma + 3 * x0, Nr)
    dt = T / Nt
    L = matrix_L(Nr, x_grid, a, b, sigma)
    I = np.eye(Nr)
    M1 = I - 0.5 * dt * L
    M2 = I + 0.5 * dt * L
    V = np.full(Nr, N)
    #Kc = 0.95 * N
    for n in reversed(range(Nt)):
        V = solve(M1, M2 @ V)
        V[0] = N
        V[-1] = 0
        t0 = n * dt
        t1 = (n + 1) * dt
        V *= shift_function(t0, t1, a, b, sigma, x0, yield_curve)
        #V = np.minimum(V, Kc)  # Call option payoff
    return np.interp(x0, x_grid, V)

## Numerical Results

### First Results

In [34]:
zcb_cir_analytical(x0, a, b, sigma, T, N), zcb_cir_cn(x0, a, b, sigma, T, Nt, Nr, N), zcb_cir_mc(x0, a, b, sigma, T, n_time, n_sim, N)

(0.9358035667520932, 0.9358051254150552, 0.9358477703548194)

In [35]:
zcb_cir_pp_analytical(x0, a, b, sigma, T, yield_curve), zcb_cir_pp_cn(x0, a, b, sigma, T, yield_curve), zcb_cir_pp_mc(x0, a, b, sigma, T, n_time, n_sim, yield_curve)

(0.9483522143781531, 0.9483548614401541, 0.9484141433161659)

In [36]:
t0, t1 = 0.0, T

In [37]:
shift_function(t0, t1, a, b, sigma, x0, yield_curve)

1.0134094889909564